# 🛠️ Notebook 2: Cricinfo — Implementation


## 🛠️ Setup

```bash
cd 07-object-oriented-design/cricinfo
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 What we'll build

We'll turn the Notebook 1 design into real, runnable Python:

1. `Ball` / `BallType` — one ball of cricket.
2. `Player` / `Team` — people and sides.
3. `Innings` — one side's batting turn, with a proper **over** view.
4. `Match` — glues two innings and decides the winner.
5. Bonus: `Commentary` as an **Observer** (a real-world pattern cricinfo itself uses for live updates).
6. A toy **1-over-per-side** match we run end-to-end, then a tiny **test suite**.

Every class is tiny on purpose — read them top-to-bottom.


## 1. `Ball` — the atomic event

Everything in cricket is a `Ball`. One ball knows three things: how many **runs** came off it, what **type** of ball it was, and whether a **wicket** fell. Those three facts drive everything else.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum

class BallType(Enum):
    LEGAL   = "legal"
    WIDE    = "wide"     # +1 penalty, does NOT count as a legal ball
    NO_BALL = "no_ball"  # +1 penalty, does NOT count as a legal ball

@dataclass
class Ball:
    runs: int = 0
    ball_type: BallType = BallType.LEGAL
    wicket: bool = False
    batter: str | None = None   # who faced it
    bowler: str | None = None   # who bowled it

    def total_runs(self) -> int:
        """Runs added to the team total for this ball."""
        extras = 1 if self.ball_type != BallType.LEGAL else 0
        return self.runs + extras

    def is_legal(self) -> bool:
        """Does this ball count toward the 6-ball-per-over limit?"""
        return self.ball_type == BallType.LEGAL

# Quick sanity checks — these are the tiny facts the rest of the code depends on.
assert Ball(4).total_runs() == 4
assert Ball(2, BallType.WIDE).total_runs() == 3        # 2 taken + 1 penalty
assert Ball(0, BallType.NO_BALL).total_runs() == 1     # just the penalty
assert Ball(0, BallType.WIDE).is_legal() is False
print("Ball rules OK ✅")


## 2. `Player` and `Team` — the people

Players have a **name** and a **role**. The role isn't used for scoring in this toy version, but it shows how easy it is to extend a dataclass later (e.g. pick "all-rounders only" for analytics).


In [ ]:
class Role(Enum):
    BATTER      = "batter"
    BOWLER      = "bowler"
    ALL_ROUNDER = "all_rounder"
    WICKETKEEPER = "wicketkeeper"

@dataclass
class Player:
    name: str
    role: Role = Role.BATTER

@dataclass
class Team:
    name: str
    players: list[Player] = field(default_factory=list)

    def add(self, name: str, role: Role = Role.BATTER) -> Player:
        p = Player(name, role)
        self.players.append(p)
        return p

india = Team("India")
india.add("Rohit")
india.add("Kohli")
india.add("Bumrah", Role.BOWLER)
print(india)


## 3. `Innings` — one side's batting turn

`Innings` owns the running totals. It doesn't care what the *other* innings is doing — that's `Match`'s job. Notice how every rule lives in **one** method:

- "A wide adds 1 run but isn't a legal ball" → already encoded in `Ball`.
- "1 over = 6 legal balls" → `over_count()`.
- "Innings ends at 10 wickets OR the over limit" → `is_complete()`.


In [ ]:
@dataclass
class Innings:
    batting: Team
    max_overs: int = 20
    runs: int = 0
    wickets: int = 0
    legal_balls: int = 0
    log: list[Ball] = field(default_factory=list)

    # -- queries ------------------------------------------------------
    def over_count(self) -> str:
        """Cricket-style overs: '12.4' = 12 complete overs + 4 balls."""
        return f"{self.legal_balls // 6}.{self.legal_balls % 6}"

    def is_complete(self) -> bool:
        return self.wickets >= 10 or self.legal_balls >= self.max_overs * 6

    def overs_played(self) -> list[list[Ball]]:
        """Group the ball log into overs (6 legal balls each)."""
        overs: list[list[Ball]] = [[]]
        legal_in_over = 0
        for b in self.log:
            overs[-1].append(b)
            if b.is_legal():
                legal_in_over += 1
                if legal_in_over == 6:
                    overs.append([])
                    legal_in_over = 0
        if not overs[-1]:
            overs.pop()
        return overs

    def summary(self) -> str:
        return f"{self.batting.name}: {self.runs}/{self.wickets} ({self.over_count()} ov)"

    # -- mutation -----------------------------------------------------
    def record(self, ball: Ball) -> None:
        if self.is_complete():
            raise RuntimeError("Innings is already complete; cannot record more balls.")
        self.log.append(ball)
        self.runs += ball.total_runs()
        if ball.is_legal():
            self.legal_balls += 1
        if ball.wicket:
            self.wickets += 1


## 4. `Match` — glue + winner

`Match` is the smallest class we have. That's a *good sign* — it means the complex rules live where they belong (in `Innings` and `Ball`), and `Match` just orchestrates.

Real cricket announces winners in a specific way:

- **Team batting second wins** → "won by X **wickets**" (wickets left in hand).
- **Team batting first wins** → "won by X **runs**".
- **Tie** → same score after both innings.


In [ ]:
@dataclass
class Match:
    team_a: Team
    team_b: Team
    max_overs: int = 20
    innings: list[Innings] = field(default_factory=list)

    def start(self, batting_first: Team) -> None:
        second = self.team_b if batting_first is self.team_a else self.team_a
        self.innings = [
            Innings(batting_first, self.max_overs),
            Innings(second,        self.max_overs),
        ]

    def current_innings(self) -> Innings:
        for inn in self.innings:
            if not inn.is_complete():
                return inn
        raise RuntimeError("Both innings are complete.")

    def target(self) -> int | None:
        """Runs the chasing side needs to win."""
        if len(self.innings) < 2:
            return None
        return self.innings[0].runs + 1

    def result(self) -> str:
        """Human-readable result — or 'in progress' if not both innings are done."""
        # First innings still running? Game is alive.
        if not self.innings[0].is_complete():
            return "in progress"
        # Second innings may also be complete if they were bowled out OR ran out of overs.
        chasing = self.innings[1]
        setting = self.innings[0]
        if chasing.runs > setting.runs:
            wickets_left = 10 - chasing.wickets
            return f"{chasing.batting.name} won by {wickets_left} wicket(s)"
        if not chasing.is_complete():
            return "in progress"
        if setting.runs > chasing.runs:
            margin = setting.runs - chasing.runs
            return f"{setting.batting.name} won by {margin} run(s)"
        return "Match tied"


## 5. A toy match, ball by ball

Let's play a **1-over-per-side** match. Small enough to follow every ball, big enough to exercise every rule.


In [ ]:
india = Team("India", [Player(n) for n in ("Rohit", "Kohli", "Bumrah")])
aus   = Team("Aus",   [Player(n) for n in ("Warner", "Smith", "Cummins")])

m = Match(india, aus, max_overs=1)
m.start(batting_first=india)

# India innings: 1, 4, W, 2, 6, 0  →  13/1 in 1 over
india_balls = [
    Ball(1),
    Ball(4),
    Ball(0, wicket=True),
    Ball(2),
    Ball(6),
    Ball(0),
]
for b in india_balls:
    m.current_innings().record(b)

print(m.innings[0].summary())
print("Target for Aus:", m.target())


In [ ]:
# Aus needs 14 to win. We'll throw in a wide to show the rule.
aus_balls = [
    Ball(0),                                   # dot ball
    Ball(0),                                   # dot ball
    Ball(1),                                   # single
    Ball(1),                                   # single
    Ball(2, ball_type=BallType.WIDE),          # 2 taken off a wide → +3 runs, NOT a legal ball
    Ball(6),                                   # six!
    Ball(2),                                   # final legal ball of the over
]
for b in aus_balls:
    m.current_innings().record(b)

print(m.innings[1].summary())
print("Result:", m.result())


### Reading the scorecard over-by-over

Let's pretty-print each over the way cricinfo itself does. This is a tiny but useful demo that the `overs_played()` grouping actually works.


In [ ]:
def render_scorecard(match: Match) -> None:
    for inn in match.innings:
        print("—", inn.summary())
        for i, over in enumerate(inn.overs_played(), start=1):
            symbols = []
            for b in over:
                if b.wicket:
                    symbols.append("W")
                elif b.ball_type == BallType.WIDE:
                    symbols.append(f"wd{b.runs}" if b.runs else "wd")
                elif b.ball_type == BallType.NO_BALL:
                    symbols.append(f"nb{b.runs}" if b.runs else "nb")
                else:
                    symbols.append(str(b.runs))
            print(f"  Over {i}: {' '.join(symbols)}")

render_scorecard(m)


## 6. 🔭 Bonus — `Commentary` as an Observer

Real cricinfo pushes live updates to millions of browsers. Every time a ball is bowled, commentary, stats, and graphs all refresh. That's the **Observer pattern**: one subject (the innings), many listeners.

Below, we wire up a simple observer list on `Innings` without touching its core logic. Notice the `Innings` class stays focused on scoring — the observers are plug-ins.


In [ ]:
from typing import Callable, Protocol

class BallObserver(Protocol):
    def on_ball(self, inn: Innings, ball: Ball) -> None: ...

# Add observer support to Innings — by monkey-patching here just to keep the
# earlier cell self-contained. In a real codebase you'd put this in the class.
def _with_observers(cls):
    original_record = cls.record
    def record(self, ball: Ball) -> None:
        original_record(self, ball)
        for obs in getattr(self, "_observers", []):
            obs(self, ball)
    def subscribe(self, observer: Callable[[Innings, Ball], None]) -> None:
        self._observers = getattr(self, "_observers", []) + [observer]
    cls.record = record
    cls.subscribe = subscribe
    return cls

_with_observers(Innings)

def live_commentary(inn: Innings, ball: Ball) -> None:
    icon = "🏏"
    if ball.wicket: icon = "🎯 OUT!"
    elif ball.runs == 6: icon = "💥 SIX!"
    elif ball.runs == 4: icon = "🔥 FOUR"
    elif ball.ball_type != BallType.LEGAL: icon = "⚠️ extra"
    print(f"[{inn.over_count()}] {icon} {inn.batting.name} now {inn.runs}/{inn.wickets}")

# Replay a fresh mini innings with the observer attached.
demo_team = Team("Demo", [Player("A"), Player("B")])
demo_inn  = Innings(demo_team, max_overs=1)
demo_inn.subscribe(live_commentary)

for b in [Ball(1), Ball(4), Ball(0, ball_type=BallType.WIDE),
          Ball(6), Ball(0, wicket=True), Ball(2), Ball(1)]:
    if not demo_inn.is_complete():
        demo_inn.record(b)

print("Final:", demo_inn.summary())


## 7. ✅ Tiny test suite

These asserts double as documentation: each one is a sentence in English about a cricket rule.


In [ ]:
# --- Ball rules ---
assert Ball(4).total_runs() == 4,                     "plain 4 is 4 runs"
assert Ball(0, BallType.WIDE).total_runs() == 1,      "a wide is 1 run"
assert Ball(3, BallType.NO_BALL).total_runs() == 4,   "no-ball + 3 taken = 4"
assert not Ball(0, BallType.WIDE).is_legal(),         "a wide is not a legal ball"

# --- Innings: over-limit termination ---
t = Team("T", [Player("x")])
inn = Innings(t, max_overs=1)
for _ in range(6):
    inn.record(Ball(1))
assert inn.is_complete(),          "6 legal balls = 1 over = innings over (max_overs=1)"
assert inn.runs == 6
assert inn.over_count() == "1.0"

# --- Innings: wides don't count toward the over ---
inn2 = Innings(t, max_overs=1)
for _ in range(3):
    inn2.record(Ball(0, BallType.WIDE))   # 3 wides, still 0 legal balls
assert inn2.legal_balls == 0
assert inn2.runs == 3
assert not inn2.is_complete()

# --- Match: chasing side wins by wickets ---
a = Team("A", [Player("a1")])
b = Team("B", [Player("b1")])
m = Match(a, b, max_overs=1)
m.start(batting_first=a)
for _ in range(6): m.current_innings().record(Ball(1))   # A: 6/0
m.current_innings().record(Ball(6))                      # B: 6
m.current_innings().record(Ball(1))                      # B: 7/0 — wins
assert m.result().startswith("B won by"), m.result()

# --- Match: defending side wins by runs ---
m2 = Match(a, b, max_overs=1)
m2.start(batting_first=a)
for _ in range(6): m2.current_innings().record(Ball(2))  # A: 12/0
for _ in range(6): m2.current_innings().record(Ball(1))  # B: 6/0
assert m2.result() == "A won by 6 run(s)", m2.result()

# --- Match: tie ---
m3 = Match(a, b, max_overs=1)
m3.start(batting_first=a)
for _ in range(6): m3.current_innings().record(Ball(1))  # A: 6/0
for _ in range(6): m3.current_innings().record(Ball(1))  # B: 6/0
assert m3.result() == "Match tied", m3.result()

print("All tests passed ✅")


## 🧪 Try it yourself

- **Batter stats + strike rotation.** Add `Batter(runs, balls_faced)` and swap strike on odd runs or at end of over.
- **Bowler stats.** Each over, assign a bowler; compute overs bowled, runs conceded, wickets — that's a bowling figure like `2.0-0-13-1`.
- **Required run rate.** In the second innings, print `(needed X off Y balls, RRR = Z)` after every ball — a great use case for the observer you wired up.
- **Byes / leg-byes.** Extend `BallType` and decide whether those runs count toward the batter's personal score.
- **DLS lite.** If `max_overs` is reduced mid-match, show how `is_complete()` and `target()` need to adapt.
